In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split



In [2]:
# Load both
train = pd.read_csv(r'C:\Users\hp\Desktop\ntg\Customer chur prediction\Dataset\customer_churn_dataset-testing-master.csv')
test = pd.read_csv(r'C:\Users\hp\Desktop\ntg\Customer chur prediction\Dataset\customer_churn_dataset-training-master.csv')



In [3]:
# Combine into one clean dataset
df = pd.concat([train, test], ignore_index=True)



In [4]:
# Drop the one NaN row (you found 1 null per column)
df.dropna(inplace=True)

# Drop CustomerID
df.drop('CustomerID', axis=1, inplace=True)

# Encode categoricals properly
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Subscription Type'] = le.fit_transform(df['Subscription Type'])
df['Contract Length'] = le.fit_transform(df['Contract Length'])



In [5]:
# Split
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()

In [7]:
lr.fit(X_train,y_train)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [8]:
from sklearn.metrics import accuracy_score
y_pred=lr.predict(X_test)
accuracy=accuracy_score(y_test,y_pred)


In [9]:
accuracy

0.8186397735595099

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [5, 10, 15],
    'min_samples_split': [5, 10],
    'criterion': ['gini']
}

# RandomizedSearchCV with memory-efficient settings
rs = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=6,        # reduced from 10 for less memory usage
    cv=3,
    scoring='roc_auc',
    n_jobs=1,        # sequential processing to avoid memory overhead (was -1)
    random_state=42,
    verbose=1
)
rs.fit(X_train, y_train)
print("Best params:", rs.best_params_)
print("Best AUC:", rs.best_score_)


Fitting 3 folds for each of 10 candidates, totalling 30 fits


c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
22 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\ens

MemoryError: Unable to allocate 3.08 MiB for an array with shape (404164,) and data type float64

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Calculate imbalance ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg / pos  # ~0.76 in your case
print(f"scale_pos_weight: {ratio:.2f}")

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,   # this is the key fix
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_proba))

scale_pos_weight: 0.80


c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:199: UserWarning: [09:17:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

         0.0       0.97      0.86      0.91     44943
         1.0       0.90      0.98      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.93      0.92      0.92    101042
weighted avg       0.93      0.93      0.92    101042

ROC-AUC: 0.9526534529810796


In [ ]:
from sklearn.metrics import roc_auc_score
proba = rs.best_estimator_.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, proba))

ROC-AUC: 0.9527707321629046


In [ ]:
from sklearn.preprocessing import StandardScaler
import os

# Create scaler and fit on training data
scaler = StandardScaler()
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


In [ ]:
import joblib

# Create models directory
os.makedirs('models', exist_ok=True)

# Save model and scaler
joblib.dump(rs.best_estimator_, 'models/churn_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("Model and scaler saved successfully!")

Model and scaler saved successfully!


# SHAP explainability
Use SHAP to open the model’s black box and show which features drive each churn prediction.

In [ ]:
import sys
!{sys.executable} -m pip install shap

   ---------------------------------------- 0.0/547.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/547.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/547.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/547.0 kB ? eta -:--:--
   ---------------------------------------- 547.0/547.0 kB 759.7 kB/s  0:00:00

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -----------------

In [ ]:
import shap

# Build a SHAP explainer for the best model found by RandomizedSearchCV
explainer = shap.TreeExplainer(rs.best_estimator_)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print("SHAP values shape:", shap_values.shape)

# Global feature importance
shap.summary_plot(shap_values, X_test, plot_type="bar")
shap.summary_plot(shap_values, X_test)

# Local explanation for one example
sample_id = 0
print("Local explanation for test sample #", sample_id)
shap.force_plot(
    explainer.expected_value,
    shap_values[sample_id],
    X_test.iloc[sample_id],
    matplotlib=True
)

In [ ]:
# Global - bar chart (which features matter most overall)
shap.summary_plot(shap_values, X_sample, plot_type="bar")

In [ ]:
# Global - bar chart (which features matter most overall)
shap.summary_plot(shap_values, X_sample, plot_type="bar")

In [ ]:
# Local - one customer explanation
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_sample.iloc[0],
    matplotlib=True
)